In [0]:
%sql
-- Set catalog and database for subsequent queries
USE CATALOG databricks_demo;

USE SCHEMA default;

In [0]:
%sql
-- Preview the first five user records
SELECT * FROM tb_users LIMIT 5;

In [0]:
# Load and flatten customer records from JSON, and register as a temp view

df = spark.read.format("json")\
    .option("multiline", "true")\
    .load("/Volumes/databricks_demo/default/files_data/sample_customers.json")\
    .selectExpr("explode(customers) as customers")

df = df.select(["customers.customer_id", "customers.email", "customers.profile"])

df.createTempView("vw_temp_customers")

In [0]:
%sql
-- Parse and nest customer profile values as a struct
CREATE OR REPLACE TEMP VIEW vw_temp_customers_parsed 
AS
SELECT
      customer_id,
      email,
      from_json(to_json(profile), schema_of_json('{"address":{"city":"Paris","country":"France","street":"06 Boulevard Victor Hugo"},"first_name":"Thomas","gender":"Male","last_name":"Lane"}')) AS profile_struct
FROM vw_temp_customers;

In [0]:
%sql
-- Display a raw profile object to check JSON structure
SELECT 
    profile
FROM vw_temp_customers
LIMIT 1;

In [0]:
%sql
-- Parse profiles into structured nested fields for repeated validation
CREATE OR REPLACE TEMP VIEW vw_temp_customers_parsed 
AS
SELECT
      customer_id,
      email,
      from_json(to_json(profile), schema_of_json('{"address":{"city":"Paris","country":"France","street":"06 Boulevard Victor Hugo"},"first_name":"Thomas","gender":"Male","last_name":"Lane"}')) AS profile_struct
FROM vw_temp_customers;

In [0]:
%sql
-- Extract un-nested profile attributes including address
SELECT
  customer_id,
  email,
  profile_struct.first_name,
  profile_struct.last_name,
  profile_struct.gender,
  profile_struct.address.country,
  profile_struct.address.city,
  profile_struct.address.street
FROM vw_temp_customers_parsed;

In [0]:
%sql
-- Select all attributes from parsed profiles
SELECT 
  customer_id, email, profile_struct.*
FROM vw_temp_customers_parsed;

In [0]:
%sql
-- Ingest order records from CSV into a managed table
CREATE OR REPLACE TABLE tb_orders
AS
SELECT *
FROM csv.`/Volumes/databricks_demo/default/files_data/orders.csv`
WITH (header = "true", inferSchema = "true", columnsTerminatedBy = ",", escape = "\"", quote = "\"")

In [0]:
%sql
-- Parse the books string as JSON and explode for one row per book per order
SELECT 
  customer_id,
  order_id,
  explode(from_json(books, 'array<struct<book_id:int,qty:int>>')) as book
FROM tb_orders
LIMIT 10;

In [0]:
%sql
-- Aggregate orders and distinct book IDs by customer
SELECT 
  customer_id,
  collect_set(order_id) as orders,
  collect_set(book.book_id) as books
FROM 
  (
    SELECT 
      customer_id,
      order_id,
      explode(from_json(books, 'array<struct<book_id:int,qty:int>>')) as book
    FROM tb_orders
  ) x
WHERE customer_id in (16,82)
GROUP BY customer_id
LIMIT 5;

In [0]:
%sql
-- Nested aggregation and flattening for books per customer
SELECT 
  customer_id,
  collect_set(order_id) as orders,
  collect_set(books) as books,
  array_distinct(flatten(collect_set(books))) as books
FROM (
    SELECT 
      customer_id,
      order_id,
      collect_set(book.book_id) as books
    FROM 
      (
        SELECT 
          customer_id,
          order_id,
          explode(from_json(books, 'array<struct<book_id:int,qty:int>>')) as book
        FROM tb_orders
      ) x
    WHERE customer_id in (16, 82)
    GROUP BY customer_id, order_id
) y 
GROUP BY customer_id;

In [0]:
%sql
-- Preview the books lookup table
SELECT * FROM tb_books LIMIT 5;

In [0]:
%sql
-- Join orders and book details to build an enriched orders table
CREATE OR REPLACE TABLE tb_orders_enriched AS
    SELECT 
      o.customer_id,
      o.order_id,  
      o.book.book_id as book_id,
      b.title as title,
      b.author as author,
      b.category as category,
      b.price as price,
      o.book.qty,
      o.book.qty * b.price as total
    FROM 
      (
        SELECT 
          customer_id,
          order_id,
          explode(from_json(books, 'array<struct<book_id:int,qty:int>>')) as book
        FROM tb_orders
      ) o
      LEFT JOIN tb_books b ON o.book.book_id = b.book_id
      ORDER BY customer_id, order_id, book.book_id;

SELECT * FROM tb_orders_enriched LIMIT 2;

In [0]:
%sql
-- Build sample_table_a with three users
CREATE OR REPLACE TABLE sample_table_a AS
SELECT 1 AS id, 'Alice' AS name UNION ALL
SELECT 2, 'Bob' UNION ALL
SELECT 3, 'Charlie';

SELECT * FROM sample_table_a;

In [0]:
%sql
-- Build sample_table_b with overlap and unique values
CREATE OR REPLACE TABLE sample_table_b AS
SELECT 2 AS id, 'Bob' AS name UNION ALL
SELECT 3, 'Charlie' UNION ALL
SELECT 4, 'Dana';

SELECT * FROM sample_table_b;

In [0]:
%sql
-- UNION: combine distinct rows from both tables
SELECT * FROM sample_table_a
UNION
SELECT * FROM sample_table_b;

In [0]:
%sql
-- UNION ALL: combine all rows including duplicates
SELECT * FROM sample_table_a
UNION ALL
SELECT * FROM sample_table_b;

In [0]:
%sql
-- INTERSECT: return rows present in both tables
SELECT * FROM sample_table_a
INTERSECT
SELECT * FROM sample_table_b;

In [0]:
%sql
-- EXCEPT: return rows present in A but not in B
SELECT * FROM sample_table_a
EXCEPT
SELECT * FROM sample_table_b;

In [0]:
%sql
-- List distinct book categories in the enriched orders table
SELECT DISTINCT category FROM tb_orders_enriched;

In [0]:
%sql
-- Pivot spend by customer across categories (multiple measures option shown)
CREATE OR REPLACE TABLE tb_spend_by_category AS
SELECT * FROM (
  SELECT 
    customer_id,
    CASE WHEN category = 'Science Fiction' THEN 'Science-Fiction' ELSE category END AS category,
    total
  FROM tb_orders_enriched
) PIVOT (
  SUM(total) AS total_sum
  FOR category IN ('Science', 'Thriller','Children','Mystery','Biography','Fiction','Comics','Science-Fiction', 'Non-Fiction', 'Travel')
);

In [0]:
%sql
-- Show a preview of the pivoted spend by category
SELECT * FROM tb_spend_by_category LIMIT 5;

In [0]:
%sql
-- Unpivot spend by category columns to long-format rows
SELECT customer_id, category, total_sum
FROM tb_spend_by_category
LATERAL VIEW EXPLODE(
  ARRAY(
    STRUCT('Science' AS category, Science AS total_sum),
    STRUCT('Thriller' AS category, Thriller AS total_sum),
    STRUCT('Children' AS category, Children AS total_sum),
    STRUCT('Mystery' AS category, Mystery AS total_sum),
    STRUCT('Biography' AS category, Biography AS total_sum),
    STRUCT('Fiction' AS category, Fiction AS total_sum),
    STRUCT('Comics' AS category, Comics AS total_sum),
    STRUCT('Science-Fiction' AS category, `Science-Fiction` AS total_sum),
    STRUCT('Non-Fiction' AS category, `Non-Fiction` AS total_sum),
    STRUCT('Travel' AS category, Travel AS total_sum)
  )
)
WHERE total_sum IS NOT NULL;